# Chat26: C1 two-session nonofficial GPU acceptance

This notebook **never authorizes official training or B-TEST**. It clones the immutable C1 release fa6a7675daae385a04c7b77e69a9de29f9e1b2f3, not the notebook branch. It uses the **previously tested** Chat25 corrected R0 restaging script from the portable migration archive, and the published C1 final native workers.

Before running: Kaggle T4x2 + internet; mount the four original datasets and the Chat26 PORTABLE migration as a Kaggle dataset (either the original ZIP bytes retained as .zipblob, or its extracted contents). Choose one PHASE: FRESH_EPOCH1 for the first Save & Run All; RESUME_EPOCH2 for a **new** Kaggle version after independently downloading, SHA-verifying and reuploading the epoch-1 archive, a filename-correct sidecar, and R9 offsite verification attestation per arm.

Default EXECUTE_ACCEPTANCE=False is CPU source/plan review only. User must explicitly turn it on to run nonofficial GPUs. Do not run the two phases consecutively in the same session.

In [ ]:
from pathlib import Path
import hashlib, json, os, io, runpy, shutil, subprocess, sys, zipfile
SHA="fa6a7675daae385a04c7b77e69a9de29f9e1b2f3"
BASE="b783eb3a470ab010785a8a24a90a666e9a3f2ed6"
PORTABLE_SHA="3f0a386ace43453921ddd84b0e5f1aef5dacb72493d11fd5fbadcfad3c974e3c"
PHASE="FRESH_EPOCH1"       # CHANGE TO "RESUME_EPOCH2" IN NEW KAGGLE VERSION
EXECUTE_ACCEPTANCE=False   # EXPLICIT USER SWITCH REQUIRED
RID={"MV00":"C1ACCEPT_MV00_20261021","MV01":"C1ACCEPT_MV01_20261021"}
ROOT=Path("/kaggle/input")
REPO=Path("/kaggle/working/CHAT26_C1_EXACT_PINNED_REPO")
PLAN=Path("/kaggle/working/CHAT26_C1_PLANS_"+PHASE)
OUT=Path("/kaggle/working/CHAT26_C1_OUTPUT_"+PHASE)
LOG=Path("/kaggle/working/CHAT26_C1_LOGS_"+PHASE)
assert PHASE in ("FRESH_EPOCH1","RESUME_EPOCH2") and type(EXECUTE_ACCEPTANCE) is bool
assert ROOT.is_dir() and not any(p.exists() for p in (REPO,PLAN,OUT,LOG))
def digest(p):
    h=hashlib.sha256()
    with Path(p).open("rb") as f:
        for b in iter(lambda:f.read(1048576),b""):h.update(b)
    return h.hexdigest()
print("NONOFFICIAL_C1_ACCEPTANCE_ONLY",PHASE,"OFFICIAL_AUTHORIZATION_FALSE; B_TEST_NONE")


In [ ]:
subprocess.run(["git","clone","--no-checkout","https://github.com/nithoukhan1/ResEMA.git",str(REPO)],check=True)
subprocess.run(["git","-C",str(REPO),"checkout","--detach",SHA],check=True)
assert subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()==SHA
REL=REPO/"research/r9_training"; C1=REL/"official_candidate"; CODE=REL/"code"
sys.path[:0]=[str(C1),str(CODE)]
from R9_OFFICIAL_RELEASE_IDENTITY_R8 import validate_checkout
assert validate_checkout(REPO,SHA)["status"]=="R8_EXACT_CHECKOUT_SHA_AND_ADDITIVE_DIFF_VERIFIED_NOT_REMOTE_PROOF"
print("EXACT_C1_RELEASE_PASS",SHA)


In [ ]:
# Execute exact tested corrected R0 restager from the migration source, never an approximation.
NAME="CHAT25_C1_R0_STALE_PATH_SAFE_RESTAGE_CORRECTED.py"
def safe_member(s):
    from pathlib import PurePosixPath
    p=PurePosixPath(s)
    return not p.is_absolute() and ".." not in p.parts and "\\" not in s
def nested_scan(blob,depth=0):
    if depth>3:return []
    out=[]
    with zipfile.ZipFile(io.BytesIO(blob)) as z:
        for m in z.infolist():
            if m.is_dir() or not safe_member(m.filename):continue
            if m.filename==NAME or m.filename.endswith("/"+NAME):
                assert m.file_size<2_000_000
                out.append(z.read(m))
            elif m.filename.lower().endswith(".zip") and m.file_size<120_000_000:
                out+=nested_scan(z.read(m),depth+1)
    return out
parts=[p.read_bytes() for p in ROOT.rglob(NAME) if p.is_file() and not p.is_symlink()]
for z in ROOT.rglob("*"):
    if z.is_file() and "DETECTION_PROJECT_MASTER_MIGRATION_CHAT1_TO_CHAT25_FOR_CHAT26_2026-10-21_PORTABLE" in z.name and z.suffix in (".zip",".zipblob"):
        assert digest(z)==PORTABLE_SHA, "PORTABLE_MIGRATION_SHA_MISMATCH"
        parts+=nested_scan(z.read_bytes())
assert parts, "MOUNT_EXISTING_CORRECTED_CHAT25_R0_SCRIPT; STOP_NO_GPU"
assert len({hashlib.sha256(b).hexdigest() for b in parts})==1,"CONFLICTING_RESTAGERS"
s=Path("/kaggle/working")/NAME;s.write_bytes(parts[0])
print("CORRECTED_RESTAGER_HASH",digest(s))
runpy.run_path(str(s),run_name="__main__")
print("CORRECTED_R0_RESTAGER_EXECUTED; R0 WILL_BE_REVALIDATED_BY_C1_PLAN_FACTORY")


In [ ]:
# Immutable SHA-based map. Only original mounted dataset files may be used as C1 inputs.
PINS={
 "runtime_yaml":("f2cc6e31f5f1be152f4dd7603a397e142963923f1d35de613eb7c61b168bf0c0",(".yaml",".yml")),
 "assignment":("87aa5e9089c6282df14ae32917946d70d9f56eaa9a5d63443985877ab9dd1ef4",(".json",)),
 "visibility":("9de034fa7df668d17effa843674c097341af9dd6461963f9e2c454c5f358b289",(".json",)),
 "pretrained_best":("620594d3560310b5d24243e1b321ce4463b0795b295891c47f6ec177e5272ac1",(".pt",)),
 "pretrained_archive":("4e3a811be5b8126e788ca4b7854e2df2b070a08e2ab8f0c6bbde8b6e1abef396",(".zip",".zipblob"))
}
MOUNTED=("d4e-r12-accepted-assets","d4e-r12-raw-archives-v1","detect-1","detection")
files=[]
for dirpath,dirs,fnames in os.walk(ROOT):
    dirs[:]=[d for d in dirs if not d.startswith(".")]
    for f in fnames:
        p=Path(dirpath)/f
        if any(m in str(p) for m in MOUNTED) and p.is_file() and not p.is_symlink():files.append(p)
INPUT={}
for key,(pin,suffixes) in PINS.items():
    found=[p for p in files if p.name.lower().endswith(suffixes) and digest(p)==pin]
    assert found, "MISSING_EXACT_PINNED_ASSET_"+key
    INPUT[key]=str(sorted(found,key=str)[0])
configs=[p for p in files if p.name=="RECOVERY_CONFIG.json" and "R7D_RESTORE_CONTEXT_SUPPORT_R1/inputs" in str(p)]
assert len(configs)==1,"R7D_ORIGINAL_RECOVERY_CONFIG_MOUNT_REQUIRED"
INPUT["config_path"]=str(configs[0])
print("SIX_INPUT_MAP_EXACT_BYTES_PASS",json.dumps(INPUT,indent=2))


In [ ]:
# Version2 must mount independently archived epoch1 bytes and external attestations.
from R9_OFFSESSION_TRANSPORT_R1 import inspected_archive
def resume_triplet(arm):
    options=[]
    for blob in ROOT.rglob("*_epoch001_G4_LOCAL_EXPORT.zip.zipblob"):
        if not blob.is_file() or arm not in str(blob):continue
        side=Path(str(blob)+".sha256")
        if not side.is_file() or side.read_text().split()!=[digest(blob),blob.name]:continue
        chk=inspected_archive(blob,sidecar=side)
        if (chk["arm"],chk["completed_epoch"],chk["terminal"])!=(arm,0,False):continue
        for a in blob.parent.rglob("*.json"):
            try:v=json.loads(a.read_text())
            except (OSError,ValueError):continue
            if (v.get("schema")=="R9_OFFSITE_CHECK_NOT_NATIVE_RESUME_ACCEPTANCE"
                and v.get("arm")==arm and v.get("completed_epoch")==0
                and v.get("zip_sha256")==digest(blob)):
                options.append((str(blob),str(side),str(a)))
    assert len(options)==1,"EXACT_ONE_VERIFIED_OFFSESSION_EPOCH1_TRIPLET_REQUIRED_"+arm
    return options[0]
RESUME={arm:resume_triplet(arm) for arm in RID} if PHASE=="RESUME_EPOCH2" else {}
print("RESUME_EVIDENCE_MOUNT_GATE",("BOTH_ARMS_PASS" if RESUME else "FRESH_NO_RESUME"))


In [ ]:
from R9_PRODUCTION_PLAN_FACTORY_C1 import create
from R9_PRODUCTION_APPROVAL_GATE_C1 import validate_acceptance,ACCEPTANCE_SCHEMA,ACCEPTANCE_PURPOSE
epoch=1 if PHASE=="FRESH_EPOCH1" else 2
mode="FRESH_FROM_EARLY" if epoch==1 else "RESUME_FROM_OUR_OWN_RUN"
arms={}
for a in RID:
    z=(None,None,None) if epoch==1 else RESUME[a]
    arms[a]={"mode":mode,"start_epoch_one_based":epoch,"target_epoch_one_based":epoch,
             "resume_archive":z[0],"resume_sidecar":z[1],"resume_external_attestation":z[2]}
request={"run_ids":RID,"arms":arms,"monotonic_session_seconds":32400,
         "minimum_epoch_seconds":1800,"reserve_export_seconds":3600}
result=create(INPUT,request,checkout=str(REPO),release_sha=SHA,output_dir=str(PLAN))
assert result["status"]=="TWO_ARM_PINNED_PLANS_CREATED_NO_GPU"
specs={a:json.loads((PLAN/(a+".json")).read_text()) for a in RID}
print("NONOFFICIAL_TWO_ARM_PLAN_PASS_NO_GPU",json.dumps(result,indent=2))


In [ ]:
if not EXECUTE_ACCEPTANCE:
    print("STOP_REVIEW_ONLY: NO_GPU_NO_AUTHORIZATION; change EXECUTE_ACCEPTANCE to True for nonofficial bounded acceptance")
else:
    import torch
    assert torch.cuda.is_available() and torch.cuda.device_count()==2
    assert all("T4" in torch.cuda.get_device_name(i).upper() for i in (0,1)),"PHYSICAL_T4_X2_REQUIRED"
    consent={"schema":ACCEPTANCE_SCHEMA,"purpose":ACCEPTANCE_PURPOSE,
             "release_commit":SHA,"frozen_science_commit":BASE,"run_ids":RID,
             "official_training_authorized":False,"b_test_access":"NONE",
             "max_completed_epoch_one_based":2}
    validate_acceptance(specs,consent,SHA)
    permit=PLAN/"C1_USER_ENABLED_NONOFFICIAL_ACCEPTANCE.json"
    with permit.open("x") as f:json.dump(consent,f,indent=2);f.write("\n")
    cmd=[sys.executable,"-u",str(C1/"R9_PRODUCTION_DUAL_SESSION_C1.py"),
         "--mv00",str(PLAN/"MV00.json"),"--mv01",str(PLAN/"MV01.json"),
         "--release-sha",SHA,"--authorization",str(permit),
         "--output-root",str(OUT),"--logs-root",str(LOG),
         "--execute-nonofficial-finalpath-acceptance"]
    subprocess.run(cmd,check=True)
    idx=json.loads((OUT/"C1_OUTPUT_INDEX.json").read_text())
    assert idx["both_arms_completed"] is True and not idx["failures"]
    assert idx["run_class"]=="NONOFFICIAL_FINALPATH_ACCEPTANCE"
    assert idx["offsite_persistence_verified"] is False
    for a,v in idx["arms"].items():
        assert v["completed_epoch_zero_based"]==epoch-1
        blob=OUT/v["zip_relative"]
        assert digest(blob)==v["zip_sha256"]
        # Never edit original sidecar. Add a byte-matched sidecar for .zipblob dataset remount.
        with Path(str(blob)+".sha256").open("x") as f:
            f.write(v["zip_sha256"]+"  "+blob.name+"\n")
        shutil.copy2(LOG/(a+"_native_stdout_stderr.log"),
                     OUT/(a+"_native_stdout_stderr.log"))
    print("BOTH_ARMS_LOCAL_G4_PASS_OFFSITE_UNVERIFIED; SAVE_KAGGLE_VERSION")


## Version boundary and evidence rules

Version 1 success is **not offsite verification**. Download original output bytes to Windows or another independent location, verify SHA256 and the G4 archive with the pinned transport code, preserve their filenames and produce independent offsite attestation JSON for each arm. Reupload these verified files as a Kaggle input dataset before running Version 2 with PHASE=RESUME_EPOCH2. Do not treat the attestation JSON itself as proof that the external step actually occurred. The second version must run in a new Kaggle session.

The C1 worker and supervisor keep all original native exports; no test labels, official approval record, or 100-epoch completion claim is generated. If any guard fails, stop and retain native logs, original artifacts and the exact traceback.